# 🎬 Nano Cinema AI Video Studio
### Local AI Video Production on 8GB VRAM

**Concept:** A complete pipeline to generate short films from text prompts using open-source models.
**Hardware Requirement:** Optimized for consumer GPUs (Tested on RTX 4060 8GB).

---

### 🚀 The Workflow
1.  **📝 Script:** `Llama-3.3-70b` (via Groq API) - Generates scenes, prompts, and narration.
2.  **🎨 Visuals:** `SDXL-Turbo` - Generates cinematic keyframes in milliseconds.
3.  **🗣️ Audio:** `EdgeTTS` - Synthesizes high-quality narration without API costs.
4.  **🎥 Video:** `LTX (Image-to-Video)` - Animates keyframes into video clips locally.
5.  **🎞️ Studio:** `Gradio` - An all-in-one Web UI to manage, edit, and render the final movie.

---
**👇 Run the cells below to initialize the environment and launch the Studio UI.**

## Envirnonment Setup 環境設置

In [ ]:
import gc
import os
import re
import sys
import glob
import time
import torch
import ffmpeg
import shutil
import asyncio
import edge_tts 
import subprocess 
import numpy as np
import gradio as gr
import pandas as pd
import imageio_ffmpeg 
from PIL import Image
from groq import Groq
import gradio as gr
from datetime import datetime
from diffusers.utils import export_to_video
from transformers import T5EncoderModel, T5Tokenizer
from diffusers import AutoPipelineForText2Image, DiffusionPipeline, LTXImageToVideoPipeline

In [ ]:
# ======================================================
# 🔧 System settings 系統核心設定 
# ======================================================
# initialization Groq Client 初始化Groq Client
try:
    groq_client = Groq()
    print("✅ Groq Client 初始化成功finished.")
except Exception as e:
    print(f"⚠️ Groq Client 初始化失敗failed : {e} (劇本生成將無法使用)")

# Set PyTorch CUDA memory allocation configuration 設定 PyTorch CUDA 記憶體分配配置
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Set your Groq API Key 設定你的 Groq  API 金鑰
API_KEY = "CHANGE_YOUR_GROQ_API_KEY"  # 更換成你的api key!!
os.environ["GROQ_API_KEY"] = API_KEY

# INITIALIZE GROQ CLIENT
# 初始化 Groq 客戶端
groq_client = Groq()

In [ ]:
# ======================================================
# 1. 設定工作目錄 
# os.path.abspath('') 會取得目前這份 ipynb 檔案所在的資料夾路徑
WORK_DIR = os.path.abspath('')

if os.path.exists(WORK_DIR):
    os.chdir(WORK_DIR)
    # 將工作目錄加入 Python 搜尋路徑
    if WORK_DIR not in sys.path:
        sys.path.append(WORK_DIR)

# 2. 定義 FFmpeg 工具路徑
# 這裡假設您的 ffmpeg.exe 放在專案根目錄下
# 如果是在 Windows，檔名是 ffmpeg.exe；Linux/Mac 則是 ffmpeg
bin_ext = ".exe" if os.name == 'nt' else ""
FFMPEG_PATH = os.path.join(WORK_DIR, f"ffmpeg{bin_ext}")
FFPROBE_PATH = os.path.join(WORK_DIR, f"ffprobe{bin_ext}")

# 檢查 FFmpeg 是否存在於專案目錄，如果不在，則假設使用者已安裝於系統環境變數中
if not os.path.exists(FFMPEG_PATH):
    print("⚠️ 專案目錄下未發現 ffmpeg 執行檔，將嘗試使用系統全域設定。")

# 3. 將當前目錄加入系統環境變數 (確保 subprocess 能呼叫到這裡的工具)
os.environ["PATH"] += os.pathsep + WORK_DIR

# 4. 設定 Hugging Face 下載路徑 (改為專案底下的 hf_cache 資料夾)
os.environ["HF_HOME"] = os.path.join(WORK_DIR, "hf_cache")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# 5. 資產存檔根目錄 (專案底下的 assets 資料夾)
ASSET_ROOT = os.path.join(WORK_DIR, "assets")
os.makedirs(ASSET_ROOT, exist_ok=True)

print("="*30)
print(f"✅ 系統就緒！")
print(f"📂 目前工作目錄 (WORK_DIR): {WORK_DIR}")
print(f"📦 模型快取路徑 (HF_HOME): {os.environ['HF_HOME']}")
print(f"🎞️ FFmpeg 路徑設定: {FFMPEG_PATH}")
print("="*30)

## Script generation tool  劇本生成工具

In [ ]:
def create_project_folder(theme):
    """根據時間與主題建立獨立資料夾"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_theme = re.sub(r'[\\/*?:"<>|]', "", theme)[:10] 
    project_dir = os.path.join(ASSET_ROOT, f"{timestamp}_{safe_theme}")
    os.makedirs(project_dir, exist_ok=True)
    print(f"🆕 已建立新專案資料夾: {project_dir}")
    return project_dir

def clean_text(text):
    if not text: return ""
    text = text.replace("**", "")
    return text.strip()

def generate_script(theme, num_scenes):
    project_dir = create_project_folder(theme)
    print(f"📝 正在使用 Llama-3.3 構思【通用型雙軌】劇本，主題：{theme}...")
    
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "你是一位兼具「皮克斯動畫導演」與「Stable Diffusion/LTX 視覺工程師」的 AI 大師。\n"
                        "請為接下來的使用者主題撰寫分鏡劇本，必須包含「靜態圖片提示詞」與「動態影片提示詞」。\n"
                        "\n"
                        "【🔥 黃金法則 1：視覺防呆 (針對 SDXL/LTX 模型特性)】\n"
                        "1. **強制單一主體**：每個場景只能有一個核心主角。禁止使用 'group', 'crowd', 'multiple' 等詞彙，除非背景是模糊的。\n"
                        "2. **物種與特徵鎖定**：\n"
                        "   - 對於動物/生物：使用 `solitary` (孤獨的) 或 `single` (單一的) 開頭。\n"
                        "   - 嚴格遵守自然特徵，除非是超現實主題，否則不要讓魚長腳，不要讓貓穿西裝。\n"
                        "3. **材質精確描述**：如果是特殊材質（如塑膠袋），必須強調 `translucent` (半透明)、`texture` (質感)。\n"
                        "\n"
                        "【🔥 黃金法則 2：雙軌 Prompt 建構公式 (請嚴格遵守)】\n"
                        "1. **##image prompt## (給畫圖模型 SDXL)**：\n"
                        "   - 關鍵：**將最重要最關鍵的元素放在前 10-20 個字元(第一句)，請直接切入主題，不要寫太多字，短而清楚為佳。**。\n"
                        "   - 專注於：**構圖、光影、材質、顏色、靜態細節、背景氛圍**。\n"
                        "   - 禁忌：不要寫複雜動作（如 running fast），不要寫時間流逝。\n"
                        "   - 範例：`Close-up portrait of a solitary golden hamster, fluffy texture, soft studio lighting, blurred background, 8k, photorealistic`\n"
                        "\n"
                        "2. **##video prompt## (給影片模型 LTX)**：\n"
                        "   - 專注於：**物理動態、運鏡軌跡、風向、持續性動作**。\n"
                        "   - 技巧：使用 `slow motion`, `drifting`, `swaying`, `breathing` 等持續性動詞。\n"
                        "   - 禁忌：不要寫畫面切換，不要寫太快或太複雜的變形。\n"
                        "   - 範例：`The hamster breathes softly, whiskers twitching. Slow camera zoom in. Smooth continuous movement.`\n"
                        "\n"
                        "【🔥 黃金法則 3：劇本靈魂 (給人類聽的旁白故事)】\n"
                        "1. **拒絕流水帳**：不要寫「這是一隻狗」，要寫「這隻狗以為自己是狼」，善用摹寫、譬喻等修辭。\n"
                        "2. **情感與哲理**：旁白要富有文學性、幽默感或哲理性，根據主題自動調整風格。\n"
                        "3. **禁止詞彙重複**：避免在不同場景中重複使用相同的詞彙或句型，也不要一直重複名稱跟類似代名詞，保持新鮮感和多樣性。\n"
                        "4. **口語化**：禁止使用疑問句結尾，請使用句號或驚嘆號，方便語音合成。\n"
                        "5. **旁白連貫性**：每個場景的旁白風格需要有連貫性和故事性，絕對不可以忽然變得很正式，也絕對不能使用類似的句型來寫不同場景的旁白。\n"
                        "\n"
                        "【輸出格式 (請嚴格遵守)】：\n"
                        "1. 以 '場景 [數字]' 開頭。\n"
                        "2. 中文視覺描述 (給人類檢查用)。\n"
                        "3. ##image prompt start## [SDXL Prompt] ##image prompt end##\n"
                        "4. ##video prompt start## [LTX Prompt] ##video prompt end##\n"
                        "5. ##旁白台詞start## [精彩故事性旁白] ##旁白台詞end##\n"
                        "6. 場景結束後輸入 '---' 分隔。"
                    )
                },
                {
                    "role": "user", 
                    "content": (
                        f"主題：'{theme}'\n"
                        f"場景數：{num_scenes}\n"
                        "請根據主題設計劇本。確保 Prompt 對應正確的模型特性（圖片求美，影片求穩）。"
                        "如果主題是具體的（如貓、狗），請給予具體的外觀設定；如果主題是抽象的（如詩歌），請將其具象化為視覺隱喻。"
                    )
                }
            ],
            max_tokens=4096, 
            temperature=0.85 # 提高創意度
        )
        
        script = response.choices[0].message.content
        print(f"✅ 劇本生成完成，長度: {len(script)} 字")
        
        # 解析邏輯 (更新為支援雙 Prompt)
        raw_scenes = re.split(r'\n\s*-{3,}\s*\n', script) 
        scene_descriptions = []
        english_descriptions = [] # Image Prompts
        video_prompts = []        # Video Prompts
        dialogue_texts = []

        for raw_scene in raw_scenes:
            if not raw_scene.strip(): continue
            
            # 1. 提取 Image Prompt
            img_match = re.search(r"##image prompt start##(.*?)##image prompt end##", raw_scene, re.DOTALL | re.IGNORECASE)
            # 以防萬一
            vis_match = re.search(r"##Visual description start##(.*?)##Visual description end##", raw_scene, re.DOTALL | re.IGNORECASE)
            
            if img_match:
                en_desc = clean_text(img_match.group(1))[:450]
            elif vis_match:
                en_desc = clean_text(vis_match.group(1))[:450]
            else:
                en_desc = "Cinematic shot, high quality, 8k, detailed"
            
            # 2. 提取 Video Prompt
            vid_match = re.search(r"##video prompt start##(.*?)##video prompt end##", raw_scene, re.DOTALL | re.IGNORECASE)
            if vid_match:
                vid_desc = clean_text(vid_match.group(1))[:450]
            else:
                # 如果沒有 Video Prompt，就用 Image Prompt 改寫 (防呆)
                vid_desc = f"Static camera, subtle motion, {en_desc[:100]}"

            # 3. 提取旁白
            dia_match = re.search(r"##旁白台詞start##(.*?)##旁白台詞end##", raw_scene, re.DOTALL | re.IGNORECASE)
            if dia_match:
                narration = clean_text(dia_match.group(1))
            else:
                narration = "..."

            # 4. 提取中文描述
            lines = raw_scene.split('\n')
            desc_zh = "場景"
            for line in lines:
                clean_line = clean_text(line)
                if "##" in clean_line or len(clean_line) < 2: continue
                if "場景" in clean_line or "描述" in clean_line:
                    desc_zh = clean_line
                    break
                if not desc_zh and len(clean_line) > 5:
                    desc_zh = clean_line
                    break
            
            scene_descriptions.append(desc_zh)
            english_descriptions.append(en_desc)
            video_prompts.append(vid_desc)
            dialogue_texts.append(narration)

        print(f"📊 已提取 {len(scene_descriptions)} 個場景")
        
        if len(english_descriptions) > 0:
            print(f"🔎 Image Prompt 檢查: {english_descriptions[0]}")
            print(f"🔎 Video Prompt 檢查: {video_prompts[0]}")

        # 回傳：劇本、場景描述、Image Prompts、Video Prompts、旁白文本、完整劇本、專案目錄
        return script, scene_descriptions, english_descriptions, video_prompts, dialogue_texts, script, project_dir

    except Exception as e:
        print(f"❌ 劇本生成發生錯誤: {e}")
        return f"Error: {e}", [], [], [], [], "", None, ""

## Image generation tool 圖片生成工具

In [ ]:
# 負面提示詞
UNIVERSAL_NEGATIVE_PROMPT = (
    "wrong subjects, clones, duplicates, "
    "mutation, deformed, distorted, disfigured, extra limbs, extra legs, extra arms, "
    "missing limbs, malformed limbs, fused fingers, too many fingers, "
    "bad anatomy, bad proportions, gross proportions, "
    "text, word, watermark, username, signature, error, jpeg artifacts, "
    "blurry, low quality, worst quality, overexposed, underexposed, "
)

# 「安全氣囊」Prompt，防止空字串導致模型崩潰。
FALLBACK_PROMPT = "Cinematic shot, high quality, photorealistic, 8k, detailed texture, dramatic lighting"

def generate_images(project_dir, english_descriptions, num_images, height, width, steps, cfg, target_scenes=None):
    """
    target_scenes: 一個包含數字的列表 (例如 [5, 7])，代表只要重跑第 5 和第 7 個場景。
                   如果為 None 或空列表，則代表「全部生成」。
    """
    
    # 1. 檢查前置條件
    if not project_dir: 
        return [], "⚠️ 請先執行劇本生成！", [], [] 
    
    # 建立圖片專屬資料夾
    img_dir = os.path.join(project_dir, "images")
    os.makedirs(img_dir, exist_ok=True)
    
    # 判斷是否需要載入模型
    print(f"🚀 正在載入 SDXL Turbo 模型...")
    
    pipe = None
    try:
        # 使用 safetensors 載入
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo", 
            torch_dtype=torch.float16, 
            use_safetensors=True
        ).to("cuda")

    except Exception as e:
        error_msg = f"❌ 模型載入失敗: {e}\n請確認您已安裝 `safetensors` 庫。"
        print(error_msg)
        return [], error_msg, [], []

    # 處理 target_scenes 邏輯
    scenes_to_process = set()
    if target_scenes:
        scenes_to_process = set(target_scenes)
        print(f"🎯 指定重繪場景: {scenes_to_process}")
    else:
        print(f"🎨 全量生成模式 (所有場景)")

    # 3. 開始生成 (或掃描舊檔)
    final_image_paths = []         # 用來回傳給 Gallery 顯示 (包含新舊)
    image_prompt_mapping = []      # 給 LTX 用 (包含新舊)
    
    total_scenes = len(english_descriptions)
    
    # 針對複雜材質，強制覆蓋使用者的參數設定，除非使用者在介面上把 steps 設得很高，否則我們使用一個「保底值」。
    safe_steps = max(steps, 2)  # 至少 2 步，確保整體細節。
    safe_cfg = max(cfg, 1.0) # 限制在 1.0 以上。
    print(f"⚙️ [系統優化參數] Steps: {safe_steps}, CFG: {safe_cfg} (為確保品質，已自動調整)")
    
    for idx, prompt in enumerate(english_descriptions):
        scene_num = idx + 1  # 1-based index (場景 1, 場景 2...)
        
        # 處理 Prompt 安全性，防止空字串
        safe_prompt = prompt if prompt and len(prompt.strip()) > 5 else FALLBACK_PROMPT
        
        # 判斷這個場景是否需要「生成」
        should_generate = (not target_scenes) or (scene_num in scenes_to_process)
        
        for i in range(num_images):
            filename = f"scene_{scene_num}_{i+1}.png"
            save_path = os.path.join(img_dir, filename)
            
            if should_generate:
                # === 執行生成 ===
                print(f" 🎬 正在生成場景 {scene_num} (第 {i+1} 張)...")
                try:
                    img = pipe(
                        prompt=safe_prompt, 
                        negative_prompt=UNIVERSAL_NEGATIVE_PROMPT,
                        height=height, 
                        width=width, 
                        num_inference_steps=safe_steps, 
                        guidance_scale=safe_cfg        
                    ).images[0]
                    img.save(save_path)
                except Exception as e:
                    print(f"  ❌ 場景 {scene_num} 生成失敗: {e}")
            else:
                # === 跳過生成，直接讀取舊檔 ===
                if not os.path.exists(save_path):
                    print(f"  ⚠️ 警告：場景 {scene_num} 的舊檔不存在 ({save_path})，可能需要重跑該場景。")

            # 無論是剛生成的，還是舊的，都要加入回傳列表
            if os.path.exists(save_path):
                final_image_paths.append(save_path)
                image_prompt_mapping.append({
                    "image_path": save_path,
                    "prompt": safe_prompt,
                    "scene_index": idx
                })

    # 4. 清理顯存
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    
    success_msg = f"✅ 處理完成！共包含 {len(final_image_paths)} 張圖片 (新生成 + 既有)。"
    print(success_msg)
    
    # 回傳：圖片路徑, 訊息, Gallery格式, Prompt對應表
    return final_image_paths, success_msg, final_image_paths, image_prompt_mapping

## Audio & Video generation tool 語音與影片模組 (Edge-TTS + LTX-Video)

In [ ]:
# 強力負向提示詞
LTX_NEGATIVE_PROMPT = (
    "worst quality, low quality, blurry, jpeg artifacts, low resolution, pixelated, "
    "morphing, warping, flickering, glitch, distortion, vibrating, static, frozen, "
    "wrong subjects, extra limbs, missing limbs, malformed, deformed, "
    "text, watermark, signature, username, error, "
)
# 模型路徑設定
CHECKPOINT_FILE = r"D:\AI_Player\hf_cache\hub\models--Lightricks--LTX-Video\snapshots\8984fa25007f376c1a299016d0957a37a2f797bb\ltxv-2b-0.9.8-distilled.safetensors"
FULL_REPO_PATH = r"D:\AI_Player\hf_cache\hub\models--Lightricks--LTX-Video\snapshots\8984fa25007f376c1a299016d0957a37a2f797bb"

# 自動鎖定 FFmpeg 執行檔
FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()
print(f"🔧 系統鎖定 FFmpeg 路徑: {FFMPEG_EXE}")

# --- 輔助函數：獲取音訊時長 ---
def get_audio_duration(audio_path):
    if not audio_path or not os.path.exists(audio_path): return 0.0
    try:
        cmd = [FFMPEG_EXE, "-i", audio_path]
        result = subprocess.run(cmd, stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True, encoding='utf-8', errors='ignore')
        match = re.search(r"Duration: (\d{2}):(\d{2}):(\d+\.\d+)", result.stderr)
        if match:
            hours, minutes, seconds = map(float, match.groups())
            return hours * 3600 + minutes * 60 + seconds
        return 0.0
    except Exception as e:
        print(f"   ⚠️ 無法讀取音訊長度: {e}")
        return 0.0

# --- 輔助函數：異步生成 TTS (單次) ---
async def _generate_tts_core(text, output_file):
    communicate = edge_tts.Communicate(text, "zh-TW-HsiaoChenNeural")
    await communicate.save(output_file)

# --- 輔助函數：帶重試機制的生成器 ---
def generate_tts_safe(text, output_file, retries=6):
    for attempt in range(retries):
        try:
            try:
                loop = asyncio.get_event_loop()
                if loop.is_running():
                    import nest_asyncio
                    nest_asyncio.apply()
                    loop.run_until_complete(_generate_tts_core(text, output_file))
                else:
                    asyncio.run(_generate_tts_core(text, output_file))
            except RuntimeError:
                new_loop = asyncio.new_event_loop()
                asyncio.set_event_loop(new_loop)
                new_loop.run_until_complete(_generate_tts_core(text, output_file))
            
            if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
                return True
            
        except Exception as e:
            print(f"   ⚠️ 連線失敗 (嘗試 {attempt+1}/{retries}): {e}")
            time.sleep(5)
            
    return False

# --- A. 語音生成函數 (穩定版) ---
def generate_audio(project_dir, dialogue_texts):
    if not project_dir: return [], None, "⚠️ 請先生成劇本！", []
    
    audio_dir = os.path.join(project_dir, "audio")
    os.makedirs(audio_dir, exist_ok=True)
    
    audio_paths = []
    audio_infos = []
    
    print(f"🗣️ 開始生成語音 (啟動慢速穩定模式)...")
    
    for idx, text in enumerate(dialogue_texts):
        if not text or not text.strip(): safe_text = "..." 
        else: safe_text = text.replace('\n', ' ').strip()
        
        save_path = os.path.join(audio_dir, f"audio_{idx+1}.mp3")
        
        # 檢查是否已存在，避免重複生成
        if os.path.exists(save_path):
            print(f"   ℹ️ 音訊已存在，跳過生成: {save_path}")
            success = True
        else:
            success = generate_tts_safe(safe_text, save_path)
            time.sleep(4) # 稍微休息

        if success:
            duration = get_audio_duration(save_path)
            audio_paths.append(save_path)
            audio_infos.append({
                "scene": f"場景 {idx+1}", 
                "text": safe_text[:15] + "...", 
                "duration": duration,   
                "path": save_path       
            })
            print(f"   ✅ 場景 {idx+1} 語音完成: {duration:.2f}秒")
        else:
            print(f"   ❌ 場景 {idx+1} 生成失敗，將使用靜音。")
            audio_paths.append(None)
            audio_infos.append({"scene": f"場景 {idx+1}", "text": "(生成失敗)", "duration": 0.0, "path": None})

    preview_path = next((p for p in audio_paths if p), None)
    return audio_infos, preview_path, f"已生成 {len([p for p in audio_paths if p])} 段語音", audio_paths

# --- 輔助函數：將秒數格式化為 SRT 時間格式 ---
def format_srt_time(seconds):
    millis = int((seconds - int(seconds)) * 1000)
    seconds = int(seconds)
    minutes = seconds // 60
    hours = minutes // 60
    minutes %= 60
    seconds %= 60
    return f"{hours:02}:{minutes:02}:{seconds:02},{millis:03}"

# --- B. 影片生成與拼接函數 (強制重繪 + 1.0s 留白 + 字幕背景) ---
def generate_video(project_dir, image_prompt_mapping, audio_paths, fps, video_prompts=None, duration_overrides=None, target_scenes=None, dialogue_texts=None):
    if not project_dir: return [], None, "⚠️ 無效的專案路徑"
    if not image_prompt_mapping: return [], None, "⚠️ 請先生成圖片"
    
    video_dir = os.path.join(project_dir, "video")
    os.makedirs(video_dir, exist_ok=True)
    
    scenes_to_process = set(target_scenes) if target_scenes else set()
    if target_scenes: print(f"🎯 指定重繪影片場景: {scenes_to_process}")
    else: print(f"🎥 全量生成/補缺模式")

    final_segments = [] 
    segment_durations = [] 
    
    for idx, item in enumerate(image_prompt_mapping):
        scene_num = idx + 1
        img_path = item["image_path"]
        
        ltx_prompt = item["prompt"] 
        if video_prompts and idx < len(video_prompts) and video_prompts[idx]:
            ltx_prompt = video_prompts[idx]
            
        raw_vid = os.path.join(video_dir, f"raw_{scene_num}.mp4")
        final_vid = os.path.join(video_dir, f"final_{scene_num}.mp4")
        
        # 🔥🔥🔥 邏輯修正區塊：決定是否生成 LTX 🔥🔥🔥
        should_generate_ltx = False
        
        if target_scenes:
            # 模式 A: 指定重繪模式
            if scene_num in scenes_to_process:
                should_generate_ltx = True
        else:
            # 模式 B: 全量補缺模式 (留空)
            if not os.path.exists(raw_vid):
                should_generate_ltx = True
            
        # -----------------------------------------------
        
        # [LTX 生成模組]
        if should_generate_ltx:
            if not os.path.exists(img_path):
                print(f" ⚠️ 跳過場景 {scene_num}: 找不到對應圖片")
                continue
            
            print(f"\n🎥 [Action!] 正在拍攝場景 {scene_num} (LTX 生成)...")
            try:
                # 清理顯存
                gc.collect()
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
                
                # 載入模型 components
                text_encoder = T5EncoderModel.from_pretrained(FULL_REPO_PATH, subfolder="text_encoder", torch_dtype=torch.bfloat16)
                tokenizer = T5Tokenizer.from_pretrained(FULL_REPO_PATH, subfolder="tokenizer")
                pipe = LTXImageToVideoPipeline.from_single_file(CHECKPOINT_FILE, text_encoder=text_encoder, tokenizer=tokenizer, torch_dtype=torch.bfloat16)
                
                # 啟用省顯存模式
                pipe.enable_model_cpu_offload()
                pipe.vae.enable_tiling()
                
                # 讀取圖片並調整大小
                img = Image.open(img_path).convert("RGB").resize((768, 512))
                
                # 生成影片 (10秒素材)
                frames = pipe(
                    image=img, 
                    prompt=ltx_prompt, 
                    negative_prompt=LTX_NEGATIVE_PROMPT, 
                    width=768, height=512, 
                    num_frames=241, 
                    num_inference_steps=53, 
                    guidance_scale=2.0
                ).frames[0]
                
                # 乒乓回放 (延長至20秒)
                reversed_frames = frames[::-1][1:] 
                frames = frames + reversed_frames 
                
                # 輸出
                export_to_video(frames, raw_vid, fps=fps)
                print(f"   ✅ LTX 畫面生成完成")
                
            except Exception as e:
                print(f"   ❌ LTX 生成失敗: {e}")
                if not os.path.exists(raw_vid): continue
                
            finally:
                # 釋放資源
                if 'pipe' in locals(): del pipe
                if 'text_encoder' in locals(): del text_encoder
                if 'tokenizer' in locals(): del tokenizer
                if 'frames' in locals(): del frames
                gc.collect()
                torch.cuda.empty_cache()

        # ----------------------------------------------------------------
        # 2. FFmpeg 音訊合成與強制對齊 (包含 +1.0s 邏輯)
        # ----------------------------------------------------------------
        if os.path.exists(raw_vid):
            audio_path = audio_paths[idx] if (audio_paths and idx < len(audio_paths)) else None
            
            # 🔥 Pacing Logic: Audio Duration + 1.0s Padding
            # 若無音訊，預設 5 秒
            target_len = 5.0 
            
            if audio_path and os.path.exists(audio_path):
                 try:
                    aud_len = get_audio_duration(audio_path)
                    if aud_len > 0:
                        # 核心邏輯：音訊長度 + 1.0秒 (固定留白)
                        target_len = aud_len + 1.0
                 except: pass
            
            # 手動覆蓋
            if duration_overrides and idx < len(duration_overrides):
                try: 
                    override = float(duration_overrides[idx])
                    if override > 0: target_len = override
                except: pass 

            print(f"   🎞️ 合成片段 {scene_num} (目標長度: {target_len:.2f}s)...")
            
            try:
                cmd = [FFMPEG_EXE, "-y", "-stream_loop", "-1", "-i", raw_vid]
                filter_str = ""
                map_args = []

                if audio_path and os.path.exists(audio_path):
                    cmd.extend(["-i", audio_path])
                    # [1:a]apad: 音訊補靜音，直到影片長度 (aud_len + 1.0) 結束
                    filter_str = f"[0:v]trim=duration={target_len},format=yuv420p[v];[1:a]apad,atrim=duration={target_len}[a]"
                    map_args = ["-map", "[v]", "-map", "[a]"]
                else:
                    filter_str = f"[0:v]trim=duration={target_len},format=yuv420p[v]"
                    map_args = ["-map", "[v]"]
                    
                cmd.extend(["-filter_complex", filter_str, *map_args, "-c:v", "libx264", "-c:a", "aac", final_vid])
                subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                segment_durations.append(target_len)
            except Exception as e:
                print(f"   ⚠️ FFmpeg 合成失敗: {e}")
                if os.path.exists(raw_vid): shutil.copy(raw_vid, final_vid)
                segment_durations.append(5.0)
        else:
            segment_durations.append(5.0)

        if os.path.exists(final_vid):
            final_segments.append(final_vid)

    # 3. 生成字幕 (SRT)
    srt_path = None
    if dialogue_texts:
        print(f"\n📝 正在生成字幕...")
        srt_path = os.path.join(project_dir, "movie.srt")
        try:
            with open(srt_path, "w", encoding="utf-8") as f:
                current_time = 0.0
                for i, text in enumerate(dialogue_texts):
                    if i >= len(segment_durations): break
                    duration = segment_durations[i] # 這是包含 +1.0s 的總長度
                    end_time = current_time + duration
                    
                    # 🔥 字幕同步邏輯：只顯示在「有聲音」的時段
                    # 總長度 = 音訊 + 1.0，所以字幕結束時間 = 總長度 - 1.0
                    if duration > 1.0:
                        sub_end_time = current_time + (duration - 1.0)
                    else:
                        sub_end_time = end_time

                    if text and text.strip() != "..." and text.strip() != "":
                        f.write(f"{i+1}\n")
                        f.write(f"{format_srt_time(current_time)} --> {format_srt_time(sub_end_time)}\n")
                        f.write(f"{text}\n\n")
                    current_time = end_time
            print(f"   ✅ 字幕檔已建立")
        except: pass

    # 4. 最終拼接與燒錄
    if not final_segments: return [], None, "❌ 無產出"

    full_movie_path = os.path.join(project_dir, "Full_Movie.mp4")
    temp_concat_path = os.path.join(project_dir, "temp_concat.mp4")
    list_txt = os.path.join(project_dir, "concat_list.txt")
    temp_srt_local = "temp_render_subs.srt"
    
    print(f"\n🎬 正在拼接 {len(final_segments)} 個片段...")
    try:
        with open(list_txt, "w", encoding='utf-8') as f:
            for v in final_segments:
                safe_path = os.path.abspath(v).replace("\\", "/")
                f.write(f"file '{safe_path}'\n")
        
        # 步驟 A: 拼接
        cmd_concat = [FFMPEG_EXE, "-y", "-f", "concat", "-safe", "0", "-i", list_txt, "-c", "copy", temp_concat_path]
        subprocess.run(cmd_concat, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # 步驟 B: 燒錄字幕 (Safe Mode)
        if srt_path and os.path.exists(srt_path):
            print(f"   ✍️ 正在燒錄字幕...")
            shutil.copy(srt_path, temp_srt_local)
            
            # 🔥 字幕樣式 (Microsoft YaHei + 降低位置 + 強制背景框)
            # Fontname=Microsoft YaHei (支援中文)
            # BackColour=&H40000000 (較深的半透明黑底，確保可見)
            # BorderStyle=3 (矩形背景框模式)
            # MarginV=14 (位置靠下)
            
            font_name = "Microsoft YaHei"
            
            # 🔥 關鍵參數：BackColour (40=75%不透明度), BorderStyle=3 (矩形框)
            style = f"Fontname={font_name},Fontsize=16,PrimaryColour=&H00FFFFFF,BackColour=&H40000000,BorderStyle=3,Outline=0,MarginV=14,Alignment=2"
            
            cmd_sub = [
                FFMPEG_EXE, "-y", "-i", temp_concat_path,
                "-vf", f"subtitles={temp_srt_local}:force_style='{style}'",
                "-c:a", "copy", "-c:v", "libx264", full_movie_path
            ]
            subprocess.run(cmd_sub, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            
            if os.path.exists(temp_concat_path): os.remove(temp_concat_path)
            if os.path.exists(temp_srt_local): os.remove(temp_srt_local)
        else:
            if os.path.exists(full_movie_path): os.remove(full_movie_path)
            os.rename(temp_concat_path, full_movie_path)

        status_msg = f"✅ 電影製作成功！已儲存於: {full_movie_path}"
    except Exception as e:
        status_msg = f"⚠️ 最終拼接失敗: {e}"
        print(status_msg)

    return final_segments, full_movie_path, status_msg

## Gradio 整合介面 (UI Code)

In [ ]:
# 自動鎖定 FFmpeg
FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()

# --- 輔助函數：獲取音訊時長 ---
def get_audio_duration(audio_path):
    if not audio_path or not os.path.exists(audio_path): return 0.0
    try:
        cmd = [FFMPEG_EXE, "-i", audio_path]
        result = subprocess.run(cmd, stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True, encoding='utf-8', errors='ignore')
        match = re.search(r"Duration: (\d{2}):(\d{2}):(\d+\.\d+)", result.stderr)
        if match:
            hours, minutes, seconds = map(float, match.groups())
            return hours * 3600 + minutes * 60 + seconds
        return 0.0
    except:
        return 0.0

# --- 輔助函數：自然排序 ---
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

# --- 輔助函數：將秒數格式化為 SRT 時間格式 ---
def format_srt_time(seconds):
    millis = int((seconds - int(seconds)) * 1000)
    seconds = int(seconds)
    minutes = seconds // 60
    hours = minutes // 60
    minutes %= 60
    seconds %= 60
    return f"{hours:02}:{minutes:02}:{seconds:02},{millis:03}"

# --- 核心功能：全域專案狀態檢查 (最高權限) ---
def check_project_status(project_dir):
    """
    掃描專案資料夾，回傳完整的資源統計字串。
    """
    if not project_dir or not os.path.exists(project_dir):
        return "⚪ 等待載入專案..."
    
    status_parts = []
    
    # 1. 劇本檢查
    if os.path.exists(os.path.join(project_dir, "script.txt")):
        status_parts.append("✅ 劇本")
    else:
        status_parts.append("❌ 無劇本")
        
    # 2. 圖片檢查
    img_dir = os.path.join(project_dir, "images")
    if os.path.exists(img_dir):
        imgs = len(glob.glob(os.path.join(img_dir, "*.png")))
        status_parts.append(f"{'✅' if imgs > 0 else '⚠️'} {imgs} 張圖")
    else:
        status_parts.append("❌ 無圖")

    # 3. 音訊檢查
    aud_dir = os.path.join(project_dir, "audio")
    if os.path.exists(aud_dir):
        auds = len(glob.glob(os.path.join(aud_dir, "*.mp3")))
        status_parts.append(f"{'✅' if auds > 0 else '⚠️'} {auds} 音訊")
    else:
        status_parts.append("❌ 無音訊")

    # 4. 影片片段檢查
    vid_dir = os.path.join(project_dir, "video")
    final_vids = 0
    if os.path.exists(vid_dir):
        final_vids = len(glob.glob(os.path.join(vid_dir, "final_*.mp4")))
        status_parts.append(f"{'✅' if final_vids > 0 else '⚠️'} {final_vids} 片段")
    else:
        status_parts.append("❌ 無片段")

    # 5. 最終成品檢查
    if os.path.exists(os.path.join(project_dir, "Full_Movie.mp4")):
        status_parts.append("🎉 成品已就緒")
        
    return " | ".join(status_parts)

# --- 核心功能：重建圖片映射表 ---
def rebuild_image_mapping(project_dir, english_descriptions):
    if not project_dir or not os.path.exists(project_dir): return []
    img_dir = os.path.join(project_dir, "images")
    if not os.path.exists(img_dir): return []
        
    print("🔄 正在重建圖片對應關係...")
    new_mapping = []
    img_files = sorted(glob.glob(os.path.join(img_dir, "*.png")), key=natural_sort_key)
    
    scene_img_map = {}
    for img_path in img_files:
        try:
            basename = os.path.basename(img_path)
            parts = basename.split("_")
            if len(parts) >= 2 and parts[1].isdigit():
                s_idx = int(parts[1]) - 1
                if s_idx not in scene_img_map:
                    scene_img_map[s_idx] = img_path
        except: pass
    
    current_prompts = english_descriptions if english_descriptions else [""] * 35
    for idx, prompt in enumerate(current_prompts):
        if idx in scene_img_map:
            new_mapping.append({
                "image_path": scene_img_map[idx],
                "prompt": prompt,
                "scene_index": idx
            })
    return new_mapping

# --- 核心功能：載入舊專案 ---
def load_existing_project(project_path):
    if not project_path or not os.path.exists(project_path):
        return [None] * 12 + [f"❌ 找不到路徑: {project_path}"]

    print(f"📂 正在載入專案: {project_path}")
    msg_log = []

    # 1. 載入劇本
    script_content = ""
    script_file = os.path.join(project_path, "script.txt")
    if os.path.exists(script_file):
        try:
            with open(script_file, "r", encoding="utf-8") as f: script_content = f.read()
            msg_log.append("✅ 劇本已載入")
        except: msg_log.append("⚠️ 劇本讀取錯誤")
    else: msg_log.append("ℹ️ 無劇本檔")

    # 2. 解析
    scene_descs, eng_descs, vid_descs, dialogues = [], [], [], []
    if script_content:
        scene_descs, eng_descs, vid_descs, dialogues, _ = parse_edited_script(script_content)
    
    # 3. 載入圖片
    img_dir = os.path.join(project_path, "images")
    gallery_images = []
    if os.path.exists(img_dir):
        gallery_images = sorted(glob.glob(os.path.join(img_dir, "*.png")), key=natural_sort_key)
        msg_log.append(f"✅ {len(gallery_images)} 張圖片")
    else: msg_log.append("⚠️ 無圖片")

    # 4. 重建 Mapping
    image_prompt_mapping = []
    if gallery_images:
        image_prompt_mapping = rebuild_image_mapping(project_path, eng_descs)

    # 5. 載入語音
    audio_dir = os.path.join(project_path, "audio")
    audio_data = [] 
    audio_paths = [] 
    
    max_len = len(dialogues) if dialogues else (len(gallery_images) if gallery_images else 35)
    audio_paths = [None] * max_len

    for i in range(max_len):
        txt = dialogues[i][:15] + "..." if (dialogues and i < len(dialogues)) else "..."
        audio_data.append([f"場景 {i+1}", txt, 0.0, 10.0])

    if os.path.exists(audio_dir):
        aud_files = sorted(glob.glob(os.path.join(audio_dir, "*.mp3")), key=natural_sort_key)
        for aud_path in aud_files:
            try:
                basename = os.path.basename(aud_path)
                parts = basename.replace("audio_", "").replace(".mp3", "")
                if parts.isdigit():
                    idx = int(parts) - 1
                    if idx < max_len:
                        audio_paths[idx] = aud_path
                        duration = get_audio_duration(aud_path)
                        audio_data[idx][2] = round(duration, 2)
                        if duration > 0:
                            vid_len = duration + 1.0 # 保持 +1.0s 邏輯
                            audio_data[idx][3] = round(vid_len, 2)
            except: pass
        msg_log.append("✅ 語音已載入")

    # 6. 載入影片
    vid_dir = os.path.join(project_path, "video")
    vid_segments = []
    full_movie = None
    if os.path.exists(vid_dir):
        vid_segments = sorted(glob.glob(os.path.join(vid_dir, "final_*.mp4")), key=natural_sort_key)
        fp = os.path.join(project_path, "Full_Movie.mp4")
        if os.path.exists(fp): full_movie = fp
        msg_log.append(f"✅ 影片已載入")

    # 🔥 呼叫全域狀態檢查
    global_status = check_project_status(project_path)

    return (
        script_content, project_path, script_content, eng_descs, vid_descs, dialogues,
        gallery_images, image_prompt_mapping, audio_data, audio_paths,
        vid_segments, full_movie, global_status 
    )

# --- 封裝：解析並同步 (修復存檔問題) ---
def parse_and_sync(raw_text, project_dir):
    # 🔥🔥🔥 1. 強制存檔：將文字框內容寫入 script.txt 🔥🔥🔥
    if project_dir and os.path.exists(project_dir):
        try:
            with open(os.path.join(project_dir, "script.txt"), "w", encoding="utf-8") as f:
                f.write(raw_text)
            print("💾 劇本已手動存檔。")
        except Exception as e:
            print(f"⚠️ 劇本存檔失敗: {e}")

    # 2. 解析內容
    scene_descs, eng_descs, vid_descs, dialogues, msg = parse_edited_script(raw_text)
    mapping = []
    if project_dir and os.path.exists(project_dir):
        mapping = rebuild_image_mapping(project_dir, eng_descs)
    
    # 3. 同步表格
    audio_data = []
    audio_paths = [None] * len(dialogues)
    
    if project_dir:
        audio_dir = os.path.join(project_dir, "audio")
        if os.path.exists(audio_dir):
            for i in range(len(dialogues)):
                potential_path = os.path.join(audio_dir, f"audio_{i+1}.mp3")
                if os.path.exists(potential_path):
                    audio_paths[i] = potential_path
    
    for i, txt in enumerate(dialogues):
        duration = 0.0
        if audio_paths[i]:
            duration = get_audio_duration(audio_paths[i])
        
        display_txt = txt[:15] + "..." if txt else "..."
        if duration > 0:
            vid_len = duration + 1.0 # 保持 +1.0s 邏輯
        else:
            vid_len = 5.0
            
        audio_data.append([f"場景 {i+1}", display_txt, round(duration, 2), round(vid_len, 2)])

    # 🔥 4. 更新全域狀態 (現在會讀到剛剛存檔的劇本了)
    global_status = check_project_status(project_dir)
    final_msg = f"{msg} | 🔄 同步完成"
    
    return (
        raw_text, eng_descs, vid_descs, dialogues, final_msg, mapping, 
        audio_data, audio_paths, global_status 
    )

# --- 解析函數 ---
def parse_edited_script(raw_text):
    if not raw_text: return [], [], [], [], "⚠️ 空劇本"
    raw_scenes = re.split(r'\n\s*-{3,}\s*\n', raw_text)
    scene_descriptions, english_descriptions, video_prompts, dialogue_texts = [], [], [], []
    
    for raw_scene in raw_scenes:
        if not raw_scene.strip(): continue
        img_match = re.search(r"##image prompt start##(.*?)##image prompt end##", raw_scene, re.DOTALL | re.IGNORECASE)
        vis_match = re.search(r"##Visual description start##(.*?)##Visual description end##", raw_scene, re.DOTALL | re.IGNORECASE)
        en_desc = clean_text(img_match.group(1))[:450] if img_match else (clean_text(vis_match.group(1))[:450] if vis_match else "Cinematic shot")

        vid_match = re.search(r"##video prompt start##(.*?)##video prompt end##", raw_scene, re.DOTALL | re.IGNORECASE)
        vid_desc = clean_text(vid_match.group(1))[:450] if vid_match else f"Static camera, subtle motion, {en_desc[:100]}"
            
        dia_match = re.search(r"##旁白台詞start##(.*?)##旁白台詞end##", raw_scene, re.DOTALL | re.IGNORECASE)
        narration = clean_text(dia_match.group(1)) if dia_match else "..."

        lines = raw_scene.split('\n')
        desc_zh = "場景"
        for line in lines:
            cl = clean_text(line)
            if "##" in cl or len(cl) < 2: continue
            if "場景" in cl or "描述" in cl: desc_zh = cl; break
        
        scene_descriptions.append(desc_zh)
        english_descriptions.append(en_desc)
        video_prompts.append(vid_desc)
        dialogue_texts.append(narration)
        
    return scene_descriptions, english_descriptions, video_prompts, dialogue_texts, f"✅ 解析完成 ({len(scene_descriptions)} 場景)"

# --- UI Wrappers ---
def generate_audio_ui_wrapper(project_dir, dialogue_texts):
    if not project_dir or not dialogue_texts: return [], None, "⚠️ 請先生成劇本！", [], check_project_status(project_dir)
    infos, preview, msg, paths = generate_audio(project_dir, dialogue_texts)
    df_data = []
    if infos:
        for info in infos:
            duration = float(info.get("duration", 0.0))
            default_vid_len = duration + 1.0 if duration > 0 else 5.0
            df_data.append([info.get("scene", "未知"), info.get("text", "..."), duration, default_vid_len])
    
    return df_data, preview, msg, paths, check_project_status(project_dir)

def generate_images_ui_wrapper(project_dir, descriptions, num, h, w, steps, cfg, target_str):
    target_list = []
    if target_str and target_str.strip():
        try: target_list = [int(x.strip()) for x in target_str.split(",") if x.strip().isdigit()]
        except: print("⚠️ 場景格式錯誤")
    paths, msg, gallery, mapping = generate_images(project_dir, descriptions, num, h, w, steps, cfg, target_scenes=target_list)
    return gallery, msg, mapping, check_project_status(project_dir)

def generate_video_ui_wrapper(project_dir, prompt_mapping, audio_paths, fps, video_prompts, duration_df, target_str, dialogue_texts):
    target_list = []
    if target_str and target_str.strip():
        try: target_list = [int(x.strip()) for x in target_str.split(",") if x.strip().isdigit()]
        except: pass
    duration_overrides = []
    if duration_df is not None:
        try:
            if isinstance(duration_df, pd.DataFrame): duration_overrides = duration_df.iloc[:, 3].tolist()
            elif isinstance(duration_df, list): duration_overrides = [row[3] for row in duration_df]
        except: pass
    
    segments, full_movie, msg = generate_video(
        project_dir, prompt_mapping, audio_paths, fps, 
        video_prompts=video_prompts, 
        duration_overrides=duration_overrides, 
        target_scenes=target_list,
        dialogue_texts=dialogue_texts 
    )
    return segments, full_movie, msg, check_project_status(project_dir)

# --- UI 主程式 ---
theme = gr.themes.Soft(
    primary_hue="blue", 
    secondary_hue="slate", 
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Noto Sans TC"), "sans-serif"]
)

css = """
footer {visibility: hidden} 
.gradio-container {max_width: 95% !important}
#script_box textarea {
    font-family: 'Consolas', 'Monaco', monospace !important; 
    font-size: 14px !important; 
    background-color: #1e1e1e !important; 
    color: #d4d4d4 !important; 
    border: 1px solid #3c3c3c !important;
    border-radius: 8px !important;
    padding: 12px !important;
    line-height: 1.5 !important;
}
.status-bar {
    font-weight: bold;
    font-size: 1.1em;
    padding: 5px;
}
"""

with gr.Blocks(title="Nano Cinema AI Video Studio", theme=theme, css=css) as demo:
    
    gr.Markdown("""
    # 🎬 Nano Cinema AI Video Studio <span style='font-size: 0.5em; color: gray; font-weight: normal;'>v1.0 Professional Edition</span>
    > **Empowering Creativity with Local AI.** > 工作流：**Script** (Llama-3) ➔ **Visuals** (SDXL-Turbo) ➔ **Audio** (EdgeTTS) ➔ **Video** (LTX) ➔ **Composition** (FFmpeg)
    """)
    
    project_dir_state = gr.State()
    script_state = gr.State()      
    eng_desc_state = gr.State()     
    vid_desc_state = gr.State()     
    dialogue_state = gr.State()     
    neg_prompt_state = gr.State()
    image_prompt_mapping_state = gr.State() 
    audio_paths_state = gr.State()

    with gr.Accordion("📂 專案管理 (Project Management)", open=True):
        with gr.Row():
            with gr.Column(scale=4):
                path_input = gr.Textbox(label="專案路徑", placeholder="例如: D:\\AI_Player\\Output_xxxx", info="貼上 Output 資料夾路徑以載入舊進度。")
            with gr.Column(scale=1):
                btn_load = gr.Button("📂 載入 / 修復專案", variant="secondary", size="lg")
        
        # 🔥 全域系統狀態
        txt_global_status = gr.Textbox(label="📊 全域資源狀態 (System Status)", value="⚪ 等待載入...", interactive=False, max_lines=1)

    with gr.Tabs() as tabs:
        # Tab 1: Script
        with gr.TabItem("1. 劇本創作 (Script)", id="tab_script"):
            with gr.Row():
                with gr.Column(scale=1, min_width=320):
                    gr.Markdown("### 📝 故事設定")
                    theme_input = gr.Textbox(label="電影主題", lines=5)
                    num_scenes = gr.Slider(label="場景數量", minimum=1, maximum=35, value=6, step=1)
                    gr.Markdown("---")
                    with gr.Row():
                        btn_script = gr.Button("✨ AI 生成劇本", variant="primary", size="lg")
                        btn_update_script = gr.Button("🔄 確認修改 & 同步", variant="secondary", size="lg")
                    lbl_script_status = gr.Label(label="執行狀態", value="就緒")

                with gr.Column(scale=2):
                    gr.Markdown("### 📜 劇本編輯器")
                    txt_script = gr.TextArea(label="Markdown 劇本內容", interactive=True, lines=28, elem_id="script_box")

        # Tab 2: Visuals
        with gr.TabItem("2. 視覺分鏡 (Visuals)", id="tab_visuals"):
            with gr.Row():
                with gr.Column(scale=1, min_width=320):
                    gr.Markdown("### 🎨 圖片生成設定")
                    img_w = gr.Number(value=768, visible=False)
                    img_h = gr.Number(value=512, visible=False)
                    with gr.Group():
                        with gr.Accordion("⚙️ 進階參數 (Advanced)", open=False):
                            img_steps = gr.Slider(
                                label="Steps (迭代步數)", 
                                minimum=1, maximum=50, value=8, step=1,
                                info="畫質精細度。數值越高細節越多，但生成速度變慢。但由於是sdxl-turbo模型，建議 4-12 即可。"
                            )
                            img_cfg = gr.Slider(
                                label="CFG Scale (提示詞權重)", 
                                minimum=0.0, maximum=15.0, value=2.0, step=0.1,
                                info="AI 對提示詞的服從度。太低畫面發散，太高畫面會崩壞。建議 1.0-4.0。"
                            )
                        gr.Markdown("### 🛠️ 局部重繪")
                        target_scenes_input = gr.Textbox(
                            label="指定場景編號", 
                            placeholder="例如: 3, 4",
                            info="若只需重畫特定幾張圖，請輸入編號 (逗號分隔)。留空則生成所有缺圖。"
                        )
                        btn_img = gr.Button("🎨 開始生成圖片", variant="primary", size="lg")
                        lbl_img_status = gr.Label(label="進度", value="等待執行...")
                with gr.Column(scale=3):
                    gal_img = gr.Gallery(label="分鏡預覽", columns=3, height="auto", object_fit="contain")

        # Tab 3: Audio
        with gr.TabItem("3. 語音與時序 (Audio)", id="tab_audio"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 🗣️ AI 語音合成")
                    btn_audio = gr.Button("生成語音 (EdgeTTS)", variant="primary")
                    gr.Markdown("---")
                    aud_preview = gr.Audio(label="試聽 (第一句)", interactive=False)
                    lbl_audio_status = gr.Label(label="狀態")
                with gr.Column(scale=2):
                    gr.Markdown("### ⏱️ 時序控制表")
                    df_audio_control = gr.Dataframe(
                        headers=["場景", "對白摘要", "語音長度(s)", "影片長度(s)"],
                        datatype=["str", "str", "number", "number"],
                        col_count=(4, "fixed"),
                        interactive=True,
                        value=[["-", "-", 0.0, 10.0]]
                    )

        # Tab 4: Render
        with gr.TabItem("4. 電影製作 (Render)", id="tab_render"):
            with gr.Row():
                with gr.Column(scale=1, min_width=320):
                    gr.Markdown("### 🎬 最終渲染")
                    vid_fps = gr.Slider(
                        label="FPS (幀率)", minimum=12, maximum=60, value=24,
                        info="影片流暢度。電影感建議 24，流暢建議 30 或 60。"
                    )
                    gr.Markdown("### 🛠️ 局部重拍 / 修復")
                    target_video_scenes_input = gr.Textbox(
                        label="指定場景編號 (操作模式)", 
                        placeholder="例如: 2, 6",
                        info="【留空】= 補齊缺少片段 | 【數字】= 強制重繪指定場景 (無視舊檔) | 【0】= 純拼接不生成"
                    )
                    btn_vid = gr.Button("🎥 開始渲染電影", variant="primary", size="lg")
                    lbl_vid_status = gr.Label(label="狀態")
                with gr.Column(scale=2):
                    vid_final = gr.Video(label="🎉 最終電影成品 (Full Movie)")
                with gr.Column(scale=1):
                    gal_vid_segments = gr.Gallery(label="分段檢視", columns=2, height="auto")

    btn_load.click(
        fn=load_existing_project, 
        inputs=[path_input], 
        outputs=[
            txt_script, project_dir_state, script_state, eng_desc_state, vid_desc_state, 
            dialogue_state, gal_img, image_prompt_mapping_state, df_audio_control, 
            audio_paths_state, gal_vid_segments, vid_final, txt_global_status
        ]
    )
    
    btn_script.click(generate_script, inputs=[theme_input, num_scenes], outputs=[txt_script, script_state, eng_desc_state, vid_desc_state, dialogue_state, script_state, project_dir_state, neg_prompt_state]).then(lambda: "✅ 劇本就緒", outputs=lbl_script_status)
    
    # 🔥 更新劇本 -> 觸發存檔 + 全域更新
    btn_update_script.click(
        fn=parse_and_sync, 
        inputs=[txt_script, project_dir_state], 
        outputs=[
            script_state, eng_desc_state, vid_desc_state, dialogue_state, 
            lbl_script_status, image_prompt_mapping_state, 
            df_audio_control, audio_paths_state, txt_global_status
        ]
    )
    
    btn_img.click(generate_images_ui_wrapper, inputs=[project_dir_state, eng_desc_state, gr.Number(1, visible=False), img_h, img_w, img_steps, img_cfg, target_scenes_input], outputs=[gal_img, lbl_img_status, image_prompt_mapping_state, txt_global_status])
    btn_audio.click(generate_audio_ui_wrapper, inputs=[project_dir_state, dialogue_state], outputs=[df_audio_control, aud_preview, lbl_audio_status, audio_paths_state, txt_global_status])
    btn_vid.click(generate_video_ui_wrapper, inputs=[project_dir_state, image_prompt_mapping_state, audio_paths_state, vid_fps, vid_desc_state, df_audio_control, target_video_scenes_input, dialogue_state], outputs=[gal_vid_segments, vid_final, lbl_vid_status, txt_global_status])

print("🚀 Nano Cinema AI Video Studio 已啟動！請打開瀏覽器...")
demo.queue() 
demo.launch(share=True, debug=True)